# Notebook 3: `z` Motif Discovery and Analysis

This notebook discovers motif flows directly in `z`, checkpoints the clean motif artifacts once, and compares the frozen and joint branches on motif quality and structure.

The defaults are tuned for a faster exploratory pass first: fewer images and no bootstrap stability filtering. You can turn the rigor back up in the parameter cell.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")
IN_COLAB = "google.colab" in sys.modules
LOCAL_REPO_DIR = Path.cwd().resolve()
if not (LOCAL_REPO_DIR / "configs" / "flow").exists() and (LOCAL_REPO_DIR.parent / "configs" / "flow").exists():
    LOCAL_REPO_DIR = LOCAL_REPO_DIR.parent

if REPO_DIR.parent.exists() and IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_ROOT = REPO_DIR if REPO_DIR.exists() else LOCAL_REPO_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import torch

try:
    from flow_circuits.data import build_cifar10_splits
    from flow_circuits.evaluation import (
        discover_motif_families,
        run_motif_gallery_experiment,
        run_motif_persistence_experiment,
        run_motif_topology_experiment,
    )
    from flow_circuits.training import load_components_from_checkpoint, load_yaml_config
except ModuleNotFoundError:
    REPO_DIR = Path("/content/model_interpretability")
    if REPO_DIR.exists() and str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(Path.cwd())], check=True)
    from flow_circuits.data import build_cifar10_splits
    from flow_circuits.evaluation import (
        discover_motif_families,
        run_motif_gallery_experiment,
        run_motif_persistence_experiment,
        run_motif_topology_experiment,
    )
    from flow_circuits.training import load_components_from_checkpoint, load_yaml_config


## Parameters


In [ ]:
CONFIG_NAME = "resnet18_z_workflow"
CONFIG_PATH = Path("configs/flow/resnet18_aligned.yaml")
DISCOVERY_MAX_IMAGES = 2000
BOOTSTRAP_ITERATIONS = 0
STABILITY_THRESHOLD = 0.0
MIN_MOTIF_LAYERS = 2
MIN_CLUSTER_SIZE = 20
MOTIF_OVERLAP_THRESHOLD = 0.50
TOP_MOTIFS_TO_RENDER = 8
BRANCHES_TO_RUN = ["frozen", "joint"]
FORCE_RERUN = False

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits") if Path("/content/drive/MyDrive").exists() else Path("artifacts/flow_circuits")
NB02_OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb02_q_validation" / CONFIG_NAME
SELECTION_ARTIFACT = NB02_OUTPUT_DIR / "selected_checkpoints.json"
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb03_z_motif_discovery_and_analysis" / CONFIG_NAME


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
base_config = load_yaml_config(CONFIG_PATH)
selection = json.loads(SELECTION_ARTIFACT.read_text(encoding="utf-8"))
FROZEN_SELECTED_CHECKPOINT = Path(selection["frozen"]["checkpoint_path"])
JOINT_SELECTED_CHECKPOINT = Path(selection["joint"]["checkpoint_path"])
loaders = build_cifar10_splits(
    data_dir=base_config["data"]["data_dir"],
    batch_size=base_config["data"]["batch_size"],
    num_workers=base_config["data"].get("num_workers", 4),
    seed=base_config["data"].get("seed", 0),
    augment_fit=base_config["data"].get("augment_fit", True),
    download=base_config["data"].get("download", True),
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 72)
print("Notebook 3 Workflow")
print("=" * 72)
print(f"Selection artifact : {SELECTION_ARTIFACT}")
print(f"Frozen checkpoint  : {FROZEN_SELECTED_CHECKPOINT}")
print(f"Joint checkpoint   : {JOINT_SELECTED_CHECKPOINT}")
print(f"Discovery images   : {DISCOVERY_MAX_IMAGES}")
print(f"Bootstrap iters    : {BOOTSTRAP_ITERATIONS}")
print(f"Stability threshold: {STABILITY_THRESHOLD}")
print(f"Branches to run    : {BRANCHES_TO_RUN}")
print(f"Device             : {device}")
print(f"Output directory   : {OUTPUT_DIR}")

_PROGRESS_STATE = {}

def _progress_logger(**event):
    checkpoint_tag = event.get("checkpoint_tag", "?")
    experiment = event.get("experiment", "?")
    stage = event.get("stage", "?")
    completed = event.get("completed")
    total = event.get("total")
    elapsed = event.get("elapsed_seconds")
    eta = event.get("eta_seconds")
    message = event.get("message", "")
    key = (checkpoint_tag, experiment, stage)
    should_print = False
    bucket = None
    if total is None or completed is None:
        should_print = True
    else:
        total = max(int(total), 1)
        completed = int(completed)
        if completed <= 1 or completed >= total:
            should_print = True
        elif total <= 10:
            should_print = True
        elif stage == "motif_node_clustering":
            bucket = completed // 8
        else:
            bucket = int((100 * completed) / total) // 20
        if bucket is not None and _PROGRESS_STATE.get(key) != bucket:
            should_print = True
            _PROGRESS_STATE[key] = bucket
    if not should_print:
        return
    parts = [f"[{checkpoint_tag}]", experiment, stage]
    if completed is not None and total is not None:
        parts.append(f"{completed}/{total}")
    if elapsed is not None:
        parts.append(f"elapsed={elapsed:.0f}s")
    if eta is not None:
        parts.append(f"eta={eta:.0f}s")
    if message:
        parts.append(message)
    print(" | ".join(parts), flush=True)

def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def _run_or_load(path: Path, label: str, fn):
    if path.exists() and not FORCE_RERUN:
        print(f"  [cache hit] {label}: {path.name}", flush=True)
        return _load_json(path)
    print(f"  [compute] {label}: {path.name}", flush=True)
    result = fn(path)
    path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"  [saved] {label}: {path.name}", flush=True)
    return result

checkpoint_map = {"frozen": FROZEN_SELECTED_CHECKPOINT, "joint": JOINT_SELECTED_CHECKPOINT}
RESULTS = {}
for tag in BRANCHES_TO_RUN:
    checkpoint_path = checkpoint_map[tag]
    print("\n" + "-" * 72)
    print(f"Branch: {tag}")
    print(f"Checkpoint: {checkpoint_path.name}")
    print("-" * 72)
    branch_dir = OUTPUT_DIR / tag
    branch_dir.mkdir(parents=True, exist_ok=True)
    print(f"Branch output dir: {branch_dir}")
    print("Existing artifacts:")
    for artifact_name in [f"{tag}_clean_motifs.json", "motif_gallery.json", "motif_persistence.json", "motif_topology.json"]:
        artifact_path = branch_dir / artifact_name
        status = "exists" if artifact_path.exists() else "missing"
        print(f"  {status:<7} {artifact_name}")
    print("Loading checkpoint components...", flush=True)
    components = load_components_from_checkpoint(checkpoint_path, device)
    motif_artifact = _run_or_load(
        branch_dir / f"{tag}_clean_motifs.json",
        "motif discovery",
        lambda cache_path, tag=tag, components=components, checkpoint_path=checkpoint_path: discover_motif_families(
            components,
            loaders["discovery"],
            device=device,
            checkpoint_tag=tag,
            max_images=DISCOVERY_MAX_IMAGES,
            nodes_per_layer=components.config["tokenization"].get("grid_size", 4) ** 2,
            bootstrap_iterations=BOOTSTRAP_ITERATIONS,
            merge_threshold=MOTIF_OVERLAP_THRESHOLD,
            min_cluster_size=MIN_CLUSTER_SIZE,
            stability_threshold=STABILITY_THRESHOLD,
            use_all_nodes=True,
            checkpoint_path=str(checkpoint_path),
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )
    gallery = _run_or_load(
        branch_dir / "motif_gallery.json",
        "motif gallery",
        lambda cache_path, tag=tag, components=components: run_motif_gallery_experiment(
            components,
            loaders["discovery"],
            device=device,
            checkpoint_tag=tag,
            motif_artifact=motif_artifact,
            max_images=DISCOVERY_MAX_IMAGES,
            topk=TOP_MOTIFS_TO_RENDER,
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )
    persistence = _run_or_load(
        branch_dir / "motif_persistence.json",
        "motif persistence",
        lambda cache_path, tag=tag: run_motif_persistence_experiment(
            motif_artifact,
            checkpoint_tag=tag,
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )
    topology = _run_or_load(
        branch_dir / "motif_topology.json",
        "motif topology",
        lambda cache_path, tag=tag: run_motif_topology_experiment(
            motif_artifact,
            checkpoint_tag=tag,
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )
    RESULTS[tag] = {
        "motif_artifact": motif_artifact,
        "motif_gallery": gallery,
        "motif_persistence": persistence,
        "motif_topology": topology,
    }

summary = {
    tag: {
        "n_motifs": RESULTS[tag]["motif_artifact"]["summary"]["n_motifs"],
        "mean_supporting_layers": RESULTS[tag]["motif_artifact"]["summary"]["mean_supporting_layers"],
        "mean_depth_span": RESULTS[tag]["motif_persistence"]["summary"]["mean_depth_span"],
        "mean_class_purity": RESULTS[tag]["motif_gallery"]["summary"]["mean_class_purity"],
    }
    for tag in RESULTS
}
(OUTPUT_DIR / "motif_analysis_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")


In [ ]:
for tag in RESULTS:
    artifact = RESULTS[tag]["motif_artifact"]
    persistence = RESULTS[tag]["motif_persistence"]
    gallery = RESULTS[tag]["motif_gallery"]
    print(f"[{tag}] n_motifs={artifact['summary']['n_motifs']} | mean_layers={artifact['summary']['mean_supporting_layers']:.3f} | mean_span={persistence['summary']['mean_depth_span']:.3f} | purity={gallery['summary']['mean_class_purity']:.3f}")
    topologies = RESULTS[tag]["motif_topology"]["summary"]["topology_counts"]
    print(f"  topology_counts={topologies}")
    print()
